# Introduction

Do higher film budgets lead to more box office revenue? Let's find out if there's a relationship using the movie budgets and financial performance data that I've scraped from [the-numbers.com](https://www.the-numbers.com/movie/budgets) on **May 1st, 2018**. 

<img src=https://i.imgur.com/kq7hrEh.png>

# Import Statements

In [79]:
import pandas as pd
import matplotlib.pyplot as plt


# Notebook Presentation

In [151]:
pd.options.display.float_format = '{:,.2f}'.format

from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

# Read the Data

In [152]:
data = pd.read_csv('cost_revenue_dirty.csv')



# Explore and Clean the Data

**Challenge**: Answer these questions about the dataset:
1. How many rows and columns does the dataset contain?
2. Are there any NaN values present?
3. Are there any duplicate rows?
4. What are the data types of the columns?

In [153]:
"""How many rows and cols?"""
print(f'row: {data.shape[0]}, cols: {data.shape[1]}')

"""Are there any NaN values present?"""
# print(data.isna())
print('Answer: There are no NaN values present in the data')

"""Are there any duplicate rows?"""
# print(data.duplicated())
print('There are no duplicate rows in the data')

"""What are the data types of the cols?"""
print(data.dtypes)

row: 5391, cols: 6
Answer: There are no NaN values present in the data
There are no duplicate rows in the data
Rank                      int64
Release_Date             object
Movie_Title              object
USD_Production_Budget    object
USD_Worldwide_Gross      object
USD_Domestic_Gross       object
dtype: object


### Data Type Conversions

**Challenge**: Convert the `USD_Production_Budget`, `USD_Worldwide_Gross`, and `USD_Domestic_Gross` columns to a numeric format by removing `$` signs and `,`. 
<br>
<br>
Note that *domestic* in this context refers to the United States.

In [154]:
data.USD_Production_Budget = data.USD_Production_Budget.str.replace('$', '')
data.USD_Production_Budget = data.USD_Production_Budget.str.replace(',', '')
data.USD_Production_Budget = pd.to_numeric(data.USD_Production_Budget)

data.USD_Worldwide_Gross = data.USD_Worldwide_Gross.str.replace('$', '')
data.USD_Worldwide_Gross = data.USD_Worldwide_Gross.str.replace(',', '')
data.USD_Worldwide_Gross = pd.to_numeric(data.USD_Worldwide_Gross)

data.USD_Domestic_Gross = data.USD_Domestic_Gross.str.replace('$', '')
data.USD_Domestic_Gross = data.USD_Domestic_Gross.str.replace(',', '')
data.USD_Domestic_Gross = pd.to_numeric(data.USD_Domestic_Gross)


**Challenge**: Convert the `Release_Date` column to a Pandas Datetime type. 

In [155]:
data.Release_Date = pd.to_datetime(data.Release_Date)
print(data.dtypes)

Rank                              int64
Release_Date             datetime64[ns]
Movie_Title                      object
USD_Production_Budget             int64
USD_Worldwide_Gross               int64
USD_Domestic_Gross                int64
dtype: object


### Descriptive Statistics

**Challenge**: 

1. What is the average production budget of the films in the data set?
2. What is the average worldwide gross revenue of films?
3. What were the minimums for worldwide and domestic revenue?
4. Are the bottom 25% of films actually profitable or do they lose money?
5. What are the highest production budget and highest worldwide gross revenue of any film?
6. How much revenue did the lowest and highest budget films make?

In [156]:
"""What is the average production budget of the films in the data set?"""
print(f'${round(data.USD_Production_Budget.mean())}')

"""What is the average worldwide gross revenue of films?"""
print(f'${round(data.USD_Worldwide_Gross.mean())}')

"""What are the minimums for worldwide and domestic revenue?"""
print(f'${data.USD_Worldwide_Gross.min()}')
print(f'${data.USD_Domestic_Gross.min()}')

"""Are the bottom 25% of films actually profitable or do they lose money?"""
bottom_25_num = round(data.shape[0]*.25)
data.sort_values(by='Rank')
bottom_movies = data[bottom_25_num:]


ww_gross = bottom_movies['USD_Worldwide_Gross'].mean()
dm_gross = bottom_movies['USD_Domestic_Gross'].mean()
pd_budget = bottom_movies['USD_Production_Budget'].mean()
result = pd_budget - (ww_gross+ dm_gross)

print(result)
print('The bottom 25% of films lose money')

"""What are the highest production budget and the highest worldwide gross revenue of any film?"""
print(f'Highest prod budget: {data['USD_Production_Budget'].max()}')
print(f'Highest gross revenue: {data['USD_Worldwide_Gross'].max()}')

"""How much revenue did the lowest and the highest budget films make?"""
data['Total Revenue'] = data['USD_Domestic_Gross'] + data['USD_Worldwide_Gross']
data_prod_sort = data.sort_values(by='USD_Production_Budget', ascending=False)

print(f'highest budget film total revenue:{data_prod_sort['Total Revenue'].iloc[0]}')
print(f'lowest budget film total revenue:{data_prod_sort['Total Revenue'].iloc[5390]}')


$31113738
$88855422
$0
$0
-100156994.64531288
The bottom 25% of films lose money
Highest prod budget: 425000000
Highest gross revenue: 2783918982
highest budget film total revenue:3544426607
lowest budget film total revenue:362082


# Investigating the Zero Revenue Films

**Challenge** How many films grossed $0 domestically (i.e., in the United States)? What were the highest budget films that grossed nothing?

In [160]:
# grossed_zero_count = data['USD_Domestic_Gross'] == 0
# grossed_zero_count.value_counts()
val_counts = data['USD_Domestic_Gross'].value_counts()
grossed_zero_count = val_counts.iloc[0]
print(grossed_zero_count)

zero_grossed_df = data[data['Total Revenue'] == 0]
zero_grossed_df.sort_values(by='USD_Production_Budget', ascending=False).head()




512


,Rank,Release_Date,Movie_Title,USD_Production_Budget,USD_Worldwide_Gross,USD_Domestic_Gross,Total Revenue
5388,96,2020-12-31,Singularity,175000000,0,0,0
5387,126,2018-12-18,Aquaman,160000000,0,0,0
5384,321,2018-09-03,A Wrinkle in Time,103000000,0,0,0
5385,366,2018-10-08,Amusement Park,100000000,0,0,0
5058,880,2015-11-12,The Ridiculous 6,60000000,0,0,0


**Challenge**: How many films grossed $0 worldwide? What are the highest budget films that had no revenue internationally?

In [166]:
zero_grossed_ww_df = data[data['USD_Worldwide_Gross'] == 0]
print(f'films that grossed $0 WW: {zero_grossed_ww_df.shape[0]}')
zero_grossed_ww_df.sort_values(by='USD_Production_Budget', ascending=False).head()


films that grossed $0 WW: 357


,Rank,Release_Date,Movie_Title,USD_Production_Budget,USD_Worldwide_Gross,USD_Domestic_Gross,Total Revenue
5388,96,2020-12-31,Singularity,175000000,0,0,0
5387,126,2018-12-18,Aquaman,160000000,0,0,0
5384,321,2018-09-03,A Wrinkle in Time,103000000,0,0,0
5385,366,2018-10-08,Amusement Park,100000000,0,0,0
5058,880,2015-11-12,The Ridiculous 6,60000000,0,0,0


### Filtering on Multiple Conditions

In [168]:
#can use .loc and bitwise operators to select data based on more than one condition
international_releases = data.loc[(data.USD_Domestic_Gross == 0) & (data.USD_Worldwide_Gross != 0)]
international_releases

,Rank,Release_Date,Movie_Title,USD_Production_Budget,USD_Worldwide_Gross,USD_Domestic_Gross,Total Revenue
71,4310,1956-02-16,Carousel,3380000,3220,0,3220
1579,5087,2001-02-11,Everything Put Together,500000,7890,0,7890
1744,3695,2001-12-31,The Hole,7500000,10834406,0,10834406
2155,4236,2003-12-31,Nothing,4000000,63180,0,63180
2203,2513,2004-03-31,The Touch,20000000,5918742,0,5918742
...,...,...,...,...,...,...,...
5340,1506,2017-04-14,Queen of the Desert,36000000,1480089,0,1480089
5348,2225,2017-05-05,Chāi dàn zhuānjiā,23000000,58807172,0,58807172
5360,4832,2017-07-03,Departure,1100000,27561,0,27561
5372,1856,2017-08-25,Ballerina,30000000,48048527,0,48048527


**Challenge**: Use the [`.query()` function](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.query.html) to accomplish the same thing. Create a subset for international releases that had some worldwide gross revenue, but made zero revenue in the United States. 

Hint: This time you'll have to use the `and` keyword.

In [169]:
data.query('USD_Domestic_Gross == 0 and USD_Worldwide_Gross != 0')

,Rank,Release_Date,Movie_Title,USD_Production_Budget,USD_Worldwide_Gross,USD_Domestic_Gross,Total Revenue
71,4310,1956-02-16,Carousel,3380000,3220,0,3220
1579,5087,2001-02-11,Everything Put Together,500000,7890,0,7890
1744,3695,2001-12-31,The Hole,7500000,10834406,0,10834406
2155,4236,2003-12-31,Nothing,4000000,63180,0,63180
2203,2513,2004-03-31,The Touch,20000000,5918742,0,5918742
...,...,...,...,...,...,...,...
5340,1506,2017-04-14,Queen of the Desert,36000000,1480089,0,1480089
5348,2225,2017-05-05,Chāi dàn zhuānjiā,23000000,58807172,0,58807172
5360,4832,2017-07-03,Departure,1100000,27561,0,27561
5372,1856,2017-08-25,Ballerina,30000000,48048527,0,48048527


### Unreleased Films

**Challenge**:
* Identify which films were not released yet as of the time of data collection (May 1st, 2018).
* How many films are included in the dataset that have not yet had a chance to be screened in the box office? 
* Create another DataFrame called data_clean that does not include these films. 

In [184]:
# Date of Data Collection

scrape_date = pd.Timestamp('2018-5-1')
unreleased_films = data[data['Release_Date'] >= scrape_date]
unreleased_films

"""How many films are included in the dataset that have not yet had a chance to be screened in the box office?"""
print('none')

"""Create another Dataframe called data_clean that does not include these films"""

print(data.shape[0])

clean_data = data.drop(index=unreleased_films.index)

# relevant_movies = data[data['Release_Date'] >= scrape_date]

clean_data

none
5391


,Rank,Release_Date,Movie_Title,USD_Production_Budget,USD_Worldwide_Gross,USD_Domestic_Gross,Total Revenue
0,5293,1915-08-02,The Birth of a Nation,110000,11000000,10000000,21000000
1,5140,1916-05-09,Intolerance,385907,0,0,0
2,5230,1916-12-24,"20,000 Leagues Under the Sea",200000,8000000,8000000,16000000
3,5299,1920-09-17,Over the Hill to the Poorhouse,100000,3000000,3000000,6000000
4,5222,1925-01-01,The Big Parade,245000,22000000,11000000,33000000
...,...,...,...,...,...,...,...
5379,1295,2017-10-02,John Wick: Chapter Two,40000000,166893990,92029184,258923174
5380,70,2017-10-03,Kong: Skull Island,185000000,561137727,168052812,729190539
5381,94,2017-12-05,King Arthur: Legend of the Sword,175000000,140012608,39175066,179187674
5382,1254,2017-12-05,Snatched,42000000,57850343,45850343,103700686


### Films that Lost Money

**Challenge**: 
What is the percentage of films where the production costs exceeded the worldwide gross revenue? 

In [196]:
total_num_movies = data.shape[0]
failed_ww = clean_data.query('USD_Production_Budget > USD_Worldwide_Gross').sort_values(by='USD_Worldwide_Gross', ascending=False)
failed_ww_count = failed_ww.shape[0]
perc = (failed_ww_count/total_num_movies)*100
round(perc)

37

# Seaborn for Data Viz: Bubble Charts

In [202]:
#import seaborn
#made on top of matplotlib

import seaborn as sbn


### Plotting Movie Releases over Time

**Challenge**: Try to create the following Bubble Chart:

<img src=https://i.imgur.com/8fUn9T6.png>



In [220]:
clean_data['Release_Date'] = pd.to_datetime(clean_data['Release_Date'], format='str').dt.date


print(clean_data['Release_Date'])

plt.figure(figsize=(8,4), dpi=200)

scatplt = sbn.scatterplot(data=clean_data, x='Release_Date', y='USD_Production_Budget',hue='USD_Worldwide_Gross', size='USD_Worldwide_Gross') 

scatplt.set(ylim=(0, 3000000000),
    xlim=(0, 450000000),
    ylabel='Revenue in $ billions',
    xlabel='Budget in $100 millions')

plt.show()

0       1915-08-02
1       1916-05-09
2       1916-12-24
3       1920-09-17
4       1925-01-01
           ...    
5379           NaT
5380           NaT
5381           NaT
5382           NaT
5383           NaT
Name: Release_Date, Length: 5384, dtype: object


OverflowError: int too big to convert

<Figure size 1600x800 with 1 Axes>

# Converting Years to Decades Trick

**Challenge**: Create a column in `data_clean` that has the decade of the release. 

<img src=https://i.imgur.com/0VEfagw.png width=650> 

Here's how: 
1. Create a [`DatetimeIndex` object](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DatetimeIndex.html) from the Release_Date column. 
2. Grab all the years from the `DatetimeIndex` object using the `.year` property.
<img src=https://i.imgur.com/5m06Ach.png width=650>
3. Use floor division `//` to convert the year data to the decades of the films.
4. Add the decades as a `Decade` column to the `data_clean` DataFrame.

### Separate the "old" (before 1969) and "New" (1970s onwards) Films

**Challenge**: Create two new DataFrames: `old_films` and `new_films`
* `old_films` should include all the films before 1969 (up to and including 1969)
* `new_films` should include all the films from 1970 onwards
* How many films were released prior to 1970?
* What was the most expensive film made prior to 1970?

# Seaborn Regression Plots

**Challenge**: Use Seaborn's `.regplot()` to show the scatter plot and linear regression line against the `new_films`. 
<br>
<br>
Style the chart

* Put the chart on a `'darkgrid'`.
* Set limits on the axes so that they don't show negative values.
* Label the axes on the plot "Revenue in \$ billions" and "Budget in \$ millions".
* Provide HEX colour codes for the plot and the regression line. Make the dots dark blue (#2f4b7c) and the line orange (#ff7c43).

Interpret the chart

* Do our data points for the new films align better or worse with the linear regression than for our older films?
* Roughly how much would a film with a budget of $150 million make according to the regression line?

# Run Your Own Regression with scikit-learn

$$ REV \hat ENUE = \theta _0 + \theta _1 BUDGET$$

**Challenge**: Run a linear regression for the `old_films`. Calculate the intercept, slope and r-squared. How much of the variance in movie revenue does the linear model explain in this case?

# Use Your Model to Make a Prediction

We just estimated the slope and intercept! Remember that our Linear Model has the following form:

$$ REV \hat ENUE = \theta _0 + \theta _1 BUDGET$$

**Challenge**:  How much global revenue does our model estimate for a film with a budget of $350 million? 